## Structured Outputs

**Week 07 — Revision Notebook**

এই notebook-এ তিনটা জিনিস আছে: **Math**, **Theory** আর আমার নিজের **Code**।
কিছু ভুলে গেলে এখানে এসে পড়ে নিলেই হবে।
প্রতিটা topic-এ একই নিয়ম: মানে → ছোট উদাহরণ → AI-তে কেন লাগে।

### সূচিপত্র (Table of Contents)

**১. MATH (Calculus I)**
- [Derivative](#s1)
- [Partial Derivative](#s2)
- [Chain Rule](#s3)

**২. THEORY (Structured Outputs)**
- [JSON Mode বনাম Structured Outputs](#s4)
- [Pydantic Validation](#s5)
- [Schema Design](#s6)
- [Parsing Failure ও Fallback](#s7)

**৩. CODE (আমার নিজের লেখা)**
- [Setup: openai install](#s8)
- [Setup: imports আর client](#s9)
- [Setup: কোন model আছে দেখা](#s10)
- [Task model](#s11)
- [Validation error demo (priority=9)](#s12)
- [get_task: retry সহ](#s13)
- [Expense model](#s14)
- [Expense-এর JSON schema](#s15)
- [extract(): generic function](#s16)
- [Final test: Expense আর Task](#s17)

**শেষে**
- [৪. আমার নোট](#s18)

<a id="p1"></a>

## ১. Math: Calculus I

Calculus মানে কোনো কিছু কত দ্রুত বদলাচ্ছে, তার হিসাব। Model-এর weight ঠিক করতে এটা লাগে।

<a id="s1"></a>

### Derivative

**মানে:** Derivative মানে একটা জায়গায় function-এর slope, অর্থাৎ x সামান্য বাড়লে f(x) কতটা বাড়ে।

**উদাহরণ:** f(x) = x² এ x=3 হলে slope = 2×3 = 6। Power rule: xⁿ এর derivative = n·xⁿ⁻¹।

**AI-তে কেন লাগে:** model-এর error কমাতে weight কোন দিকে বদলাতে হবে, derivative সেটা বলে দেয়।

### Derivative: slope function আর সোজা line-এর slope

In [1]:
def f(x):
    return 3 * x + 2

def slope(f, x, h=0.0001):
    return(f(x+h)-f(x))/h

print(slope(f, 5))
print(slope(f, 100))

2.9999999999930083
3.0000000003838068


### Derivative: x² এ x=3 হলে slope প্রায় 6

In [2]:
print(slope(lambda x: x ** 2, 3))   # প্রায় 6

6.000100000012054


### Derivative: h বদলালে উত্তর কেমন হয় (f(x) = x³, x = 2, ঠিক উত্তর 12)

In [3]:
def f(x):
    return x ** 3

def slope(f, x, h):
    return (f(x + h) - f(x)) / h

for h in [1, 0.1, 0.0001, 1e-8, 1e-14]:
    print(h, slope(f, 2, h))

1 19.0
0.1 12.61000000000001
0.0001 12.000600010022566
1e-08 11.999999927070348
1e-14 12.256862191861728


h বেশি বড় বা বেশি ছোট হলে উত্তর ভুল হয়।

<a id="s2"></a>

### Partial Derivative

**মানে:** একাধিক variable থাকলে একটা বদলাই, বাকিগুলো সাধারণ সংখ্যা ধরি।

**উদাহরণ:** f(x,y) = x² + 3y, x=2, y=5: ∂f/∂x = 2x = 4, ∂f/∂y = 3।

**AI-তে কেন লাগে:** model-এ হাজার হাজার weight থাকে, প্রতিটার আলাদা slope লাগে।

### Partial Derivative: code দিয়ে মিলিয়ে দেখি

In [4]:
def f2(x, y):
    return x ** 2 + 3 * y

# x বদলাই, y = 5 সাধারণ সংখ্যা
print("∂f/∂x ≈", round(slope(lambda x: f2(x, 5), 2, 0.0001), 3))   # প্রায় 4

# y বদলাই, x = 2 সাধারণ সংখ্যা
print("∂f/∂y ≈", round(slope(lambda y: f2(2, y), 5, 0.0001), 3))   # প্রায় 3

∂f/∂x ≈ 4.0
∂f/∂y ≈ 3.0


<a id="s3"></a>

### Chain Rule

**মানে:** function-এর ভেতরে function থাকলে বাইরের আর ভেতরের derivative গুণ করি।

**উদাহরণ:** f(x) = (2x+1)², x=1: বাইরে 2(2x+1) = 6, ভেতরে 2, গুণ করে 12।

**AI-তে কেন লাগে:** neural network-এ layer-এর ভেতরে layer থাকে, তাই backpropagation-এ এটা লাগে।

### Chain Rule: code দিয়ে মিলিয়ে দেখি

In [5]:
def inner(x):
    return 2 * x + 1

def outer(u):
    return u ** 2

def composed(x):
    return outer(inner(x))

print(round(slope(composed, 1, 0.0001), 3))   # প্রায় 12
print(2 * (2 * 1 + 1) * 2)                    # হাতের হিসাব: বাইরে 6 × ভেতরে 2 = 12

12.0
12


<a id="p2"></a>

## ২. Theory: Structured Outputs

Structured Output মানে LLM-এর উত্তর একটা ঠিক করা ছাঁচে (JSON) পাওয়া। এতে program সহজে উত্তর পড়তে পারে।

<a id="s4"></a>

### JSON Mode বনাম Structured Outputs

**মানে:** দুটোই JSON দেয়, কিন্তু নিশ্চয়তা আলাদা।

| | JSON Mode (`json_object`) | Structured Outputs (`json_schema`) |
|---|---|---|
| কী নিশ্চিত করে | শুধু valid JSON | output schema-র সাথে মিলবে |
| field-এর নাম ও type | ভুল হতে পারে | schema অনুযায়ী ঠিক থাকে |

**উদাহরণ:** `{"priority": "high"}` valid JSON, কিন্তু priority-র type `int` হওয়ার কথা ছিল। JSON Mode এটা আটকায় না।

সব provider ও model এটা support করে না, docs দেখে নিতে হয়।

আমরা এখন JSON Mode ব্যবহার করেছি, তাই Pydantic validation লাগে।

**AI-তে কেন লাগে:** program-এ LLM-এর উত্তর ব্যবহার করতে হলে নির্দিষ্ট ছাঁচে পাওয়া জরুরি।

<a id="s5"></a>

### Pydantic Validation

**মানে:** LLM-এর output সরাসরি ব্যবহার না করে আগে ছাঁচে মিলিয়ে দেখা।

**উদাহরণ:** priority 1 থেকে 5 হতে হবে, 9 এলে `ValidationError`।

**AI-তে কেন লাগে:** ভুল data database বা backend-এ গেলে সমস্যা হয়।

<a id="s6"></a>

### Schema Design

**মানে:** Schema হলো LLM-কে দেওয়া ছাঁচ: কোন field থাকবে, কোন type হবে।

**নিয়ম:**
1. field-এর নাম পরিষ্কার রাখো।
2. সীমিত মানের জন্য `Literal` দাও। সংখ্যার সীমার জন্য `Field(ge, le, gt)` দাও।
3. যা না থাকলেও চলে, সেটা optional রাখো।
4. prompt আর schema-র field-এর নাম হুবহু এক রাখো (বা schema থেকেই prompt বানাও)।

**উদাহরণ:** `category: Literal["food", "transport", "bills"]`, `priority: int = Field(ge=1, le=5)`।
`ge` = সমান বা বড়, `le` = সমান বা ছোট, `gt` = শুধু বড়।

**AI-তে কেন লাগে:** schema যত পরিষ্কার, model-এর ভুল করার জায়গা তত কম।

<a id="s7"></a>

### Parsing Failure ও Fallback

**মানে:** LLM ভুল format-এ উত্তর দিলে program যেন না ভাঙে, তার পরিকল্পনা।

**ক্রম:** validate → retry → fallback

fallback মানে: `None`, ডিফল্ট মান, বা মানুষের কাছে পাঠানো।

**উদাহরণ:** `get_task` মোট ২ বার চেষ্টা করে (`retries=1`)। তবুও ভুল হলে `None` ফেরত দেয়।

**AI-তে কেন লাগে:** LLM-এর output সবসময় ঠিক হয় না, তাই ভুল হলে কী করব, তা আগে থেকে ঠিক রাখতে হয়।

<a id="p3"></a>

## ৩. Code: আমার নিজের লেখা

এখানের সব code আমার নিজের লেখা। ক্রম: setup → model → validation → retry → generic function।

<a id="s8"></a>

### Setup: openai install

প্রথমবার `openai` package লাগে।

In [6]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


<a id="s9"></a>

### Setup: imports আর client

Groq-এর API key `GROQ_API_KEY` environment variable থেকে পড়া হয়, code-এ লেখা নেই।

In [ ]:
from pydantic import BaseModel, ValidationError, Field
import os, json
from openai import OpenAI

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",  
    api_key=os.environ["GROQ_API_KEY"],
)

<a id="s10"></a>

### Setup: কোন model আছে দেখা

In [8]:
for m in client.models.list().data:
    print(m.id)


whisper-large-v3-turbo
whisper-large-v3
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi
allam-2-7b
openai/gpt-oss-120b
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b


<a id="s11"></a>

### Task model

`Field(ge=1, le=5)` মানে priority 1 থেকে 5-এর মধ্যে হতে হবে।

In [9]:
class Task(BaseModel):
    title: str = Field(min_length=1)
    priority: int = Field(ge =1, le=5)
    done: bool

<a id="s12"></a>

### Validation error demo (priority=9)

priority=9 দিলে `ValidationError` আসে।

In [10]:
from pydantic import BaseModel, Field, ValidationError

class Task(BaseModel):
    title: str = Field(min_length=1)
    priority: int = Field(ge=1, le=5)   # ge = এর সমান বা বড়, le = এর সমান বা ছোট
    done: bool

bad = '{"title": "Report", "priority": 9, "done": false}'

try:
    Task.model_validate_json(bad)
except ValidationError as e:
    print(e)

1 validation error for Task
priority
  Input should be less than or equal to 5 [type=less_than_equal, input_value=9, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


<a id="s13"></a>

### get_task: retry সহ

ভুল format এলে আবার চেষ্টা করে। শেষেও ভুল হলে `None` দেয়।

In [7]:
PROMPT = """Return ONLY valid JSON with keys: title (string), priority (int 1-5), done (bool)."""



def get_task(text ,retries = 1, models = "openai/gpt-oss-20b"):
    prompt = f"{PROMPT}\nTask: {text}"
    for attempt in range(retries+1):
        response = client.chat.completions.create(
            model= models,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        raw = response.choices[0].message.content
        try:
            return Task.model_validate_json(raw)
        except ValidationError as e:
            print("attempt", attempt+1, "failde:", e)
    return None

print(get_task("কাল সকালের মধ্যে invoice পাঠাতে হবে, তেমন জরুরি না"))

title='কাল সকালের মধ্যে invoice পাঠাতে হবে, তেমন জরুরি না' priority=3 done=False


<a id="s14"></a>

### Expense model

`Literal` দিয়ে category শুধু তিনটা মানের একটা রাখা হয়েছে।

In [11]:
from typing import Literal

class Expense(BaseModel):
    item: str =Field(min_length=1)
    amount: float = Field(ge=0)     # শূন্যের বেশি হতে হবে
    category: Literal["food", "transport", "bills"]   # শুধু "food", "transport", "bills" এর একটা হতে হবে


PROMPT = """Return ONLY valid JSON with keys: item (string), amount, category: Literal["food", "transport", "bills"]."""



def get_Expense(text ,retries = 1, models = "openai/gpt-oss-20b"):
    prompt = f"{PROMPT}\nTask: {text}"
    for attempt in range(retries+1):
        response = client.chat.completions.create(
            model= models,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        raw = response.choices[0].message.content
        try:
            return Expense.model_validate_json(raw)
        except ValidationError as e:
            print("attempt", attempt+1, "failde:", e)
    return None

print(get_Expense("আজ রিকশা ভাড়া ৫০ টাকা দিলাম"))
 

item='rickshaw fare' amount=50.0 category='transport'


<a id="s15"></a>

### Expense-এর JSON schema

Pydantic model থেকে JSON schema বের করে দেখা।

In [13]:
print(Expense.model_json_schema())

{'properties': {'item': {'minLength': 1, 'title': 'Item', 'type': 'string'}, 'amount': {'minimum': 0, 'title': 'Amount', 'type': 'number'}, 'category': {'enum': ['food', 'transport', 'bills'], 'title': 'Category', 'type': 'string'}}, 'required': ['item', 'amount', 'category'], 'title': 'Expense', 'type': 'object'}


<a id="s16"></a>

### extract(): generic function

যেকোনো Pydantic model দিলেই কাজ করে। Schema থেকেই prompt বানায়।

In [14]:
import json

def extract(text, schema, retries=1, model="openai/gpt-oss-20b"):
    schema_text = json.dumps(schema.model_json_schema())
    prompt = f"""Return ONLY valid JSON that matches this JSON schema:
{schema_text}

Text: {text}"""

    for attempt in range(retries + 1):
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        raw = response.choices[0].message.content
        try:
            return schema.model_validate_json(raw)     # ফাঁক ১
        except ValidationError as e:
            print("attempt", attempt + 1, "failed:", e)
    return None

<a id="s17"></a>

### Final test: Expense আর Task

একই `extract()` দিয়ে দুটো আলাদা model পরীক্ষা।

In [22]:
print(extract("কাল invoice পাঠাতে হবে", Task))
print(extract("আজ রিকশা ভাড়া ৫০ টাকা দিলাম", Expense))  # ফাঁক ২

title='কাল invoice পাঠাতে হবে' priority=1 done=False
item='রিকশা ভাড়া' amount=50.0 category='transport'


<a id="p4"></a>
<a id="s18"></a>

## ৪. আমার নোট

নিজে লিখে ভরো।

### Derivative মানে কী


### LLM output validate করতে হয় কেন


### আজকের সবচেয়ে বড় ভুল